<a href="https://colab.research.google.com/github/anuradha-gh/AI-Powered-Granular-Access-Control-for-SaaS-Applications-/blob/Access-Classifier/Access_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @ Initial Setup and Installations
!pip install pandas scikit-learn xgboost -q

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
import re
from datetime import datetime
from google.colab import drive

print("Libraries installed successfully.")

# @ Load and Preprocess Real Dataset
# --- Configuration ---
DATA_PATH = '/content/drive/MyDrive/cybersecurity_threat_detection_logs.csv'

# These are the correct column names for this dataset
ACTION_COLUMN_NAME = 'action'
THREAT_LABEL_COLUMN_NAME = 'threat_label'
# ---------------------

try:
    drive.mount('/content/drive')
    df = pd.read_csv(DATA_PATH)
    print(f"Successfully loaded {len(df)} records from {DATA_PATH}")

    # --- Preprocessing ---
    # 1. Handle NaNs
    cat_cols = ['protocol', THREAT_LABEL_COLUMN_NAME, 'log_type', 'user_agent', 'request_path', ACTION_COLUMN_NAME]
    for col in cat_cols:
        if col in df.columns: df[col] = df[col].fillna('Unknown')
    num_cols = ['bytes_transferred']
    for col in num_cols:
         if col in df.columns: df[col] = df[col].fillna(0)

    # 2. Time Features
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour_of_day'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.dayofweek
    else:
        df['hour_of_day'] = np.random.randint(0, 24, size=len(df))
        df['day_of_week'] = np.random.randint(0, 7, size=len(df))

    # 3. Target Variable (y)
    df['outcome'] = df[ACTION_COLUMN_NAME].apply(lambda x: 1 if str(x).lower() == 'allowed' else 0)

    # 4. 'label_numeric' Feature
    df['label_numeric'] = df[THREAT_LABEL_COLUMN_NAME].apply(lambda x: 1 if str(x).lower() == 'suspicious' else 0)

    print("Data preprocessing complete.")

    # --- Sample the data for this demo ---
    sample_size = min(len(df), 100000)
    df_sample = df.sample(n=sample_size, random_state=42)
    print(f"Using a sample of {sample_size} records for this demo.")

except FileNotFoundError:
    print(f"ERROR: File not found at '{DATA_PATH}'.")
    df_sample = pd.DataFrame()
